In [1]:
import os

In [2]:
from dotenv import load_dotenv
load_dotenv()
os.environ["LANGCHAIN_TRACING"]="true"
os.environ["LANGCHAIN_ENDPOINT"]="https://api.smith.langchain.com"

In [3]:
LANGCHAIN_API_KEY =os.getenv("LANGCHAIN_API_KEY")
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")

In [4]:
os.environ["LANGCHAIN_API_KEY"]=LANGCHAIN_API_KEY
os.environ["GOOGLE_API_KEY"]=GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"]=TAVILY_API_KEY
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

c:\Users\sama\OneDrive\Desktop\lang_graph\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
LANGCHAIN_PROJECT

'lang_graph'

## SIMPLE AI ASSISTANT

In [7]:
while True:
    question = input("Tell your query?. Press quit to discontinue")
    if question!="quit":
        print(llm.invoke(question).content)
    else:
        print("Good Bye")
        break

Hi there! How can I help you today?
AI is the simulation of human intelligence processes by machines, enabling them to learn, reason, and problem-solve. It allows computers to perform tasks traditionally requiring human cognitive abilities.
Good Bye


In [8]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

## RAG with LCEL

In [9]:
store={}

In [10]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [11]:
config = {"configurable": {"session_id": "firstchat"}}

In [12]:
model_with_memory = RunnableWithMessageHistory(llm,get_session_history)

c:\Users\sama\OneDrive\Desktop\lang_graph\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
model_with_memory.invoke(("Hi! I am Sakshi "),config=config).content

"Hi Sakshi! Nice to meet you. I'm an AI, and I'm here to help.\n\nWhat can I do for you today?"

In [14]:
model_with_memory.invoke(("Tell me what is my name ?"),config = config).content

'Your name is **Sakshi**! You told me that yourself in our very first interaction. 😊\n\nIs there something specific you wanted to know or talk about regarding your name?'

In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough , RunnableLambda
from langchain_core.output_parsers import StrOutputParser

### Reading the txt files from source directory

loader = DirectoryLoader('../data', glob="./*.txt", loader_cls=TextLoader)
docs = loader.load()

### Creating Chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len
)
new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

###  BGE Embddings

'''from langchain.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)
'''

### Creating Retriever using Vector DB

db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [ ]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )


In [ ]:
question ="what is llama3? can you highlight 3 important points?"
print(retrieval_chain.invoke(question))

## Tools and Agents

In [15]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [16]:
api_wrapper = WikipediaAPIWrapper(top_k_results = 5, doc_content_chars_max=100)

In [17]:
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [18]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [19]:
tool.name

'wikipedia'

In [20]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [22]:
tool.return_direct

False

In [23]:
print(tool.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


## Youtube Search Tool

In [35]:

from langchain_community.tools import YouTubeSearchTool

In [36]:

tool=YouTubeSearchTool()

In [37]:

tool.name

'youtube_search'

In [1]:

tool.description

NameError: name 'tool' is not defined

In [ ]:

tool.run("sunny savita")
#run pip install youtube_search to get the desired result

In [ ]:

from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:

tool = TavilySearchResults()

In [ ]:

tool.invoke({"query": "What happened in the latest burning man floods"})